> 在线讲义：[第 13 章 · 线性倒立摆与 ZMP：双足机器人为什么要不停地迈步](https://wikieden.github.io/robot-motion-rl/lessons/13-lipm-zmp.html) · 本书《机器人运控与强化学习》

# 第 13 章 · 线性倒立摆与 ZMP：双足机器人为什么要不停地迈步

配套讲义：仓库里 `robot-motion-rl/notes/lessons/13-lipm-zmp.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。只用 NumPy，不需要 GPU。练习题不在这里算，先在讲义里自己做。

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)  # 打印保留 4 位小数，不用科学计数法
print("numpy", np.__version__)

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

（质心高度、偏移、速度都是编的）

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第二步 ω | √(g/z_c) | `omega(0.8)` | 3.5 |
| 第二步 ZMP | x − ẍ/ω² | `zmp_from_accel(0, 1, w)` | −0.0816 |
| 第三步 发散 | cosh 解 | `lipm_state(0.05, 0, 0, w, t)` | 0.053、0.080、0.148 m |
| 第四步 捕获点 | x + ẋ/ω | `capture_point(0, 0.5, w)` | 0.1429；2 s 后速度 ≈ 0 |

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

**函数 `omega`**：LIPM 的固有频率 ω = √(g / z_c)，z_c 是质心高度。

In [ ]:
def omega(z_c, g=9.8):
    """LIPM 的固有频率 ω = √(g / z_c)，z_c 是质心高度。"""
    return float(np.sqrt(g / z_c))

**函数 `lipm_state`**：ẍ = ω²(x − p) 的解析解：x = p + (x0−p)·cosh(ωt) + (v0/ω)·sinh(ωt)，返回 (x, ẋ)。

In [ ]:
def lipm_state(x0, v0, p, w, t):
    """ẍ = ω²(x − p) 的解析解：x = p + (x0−p)·cosh(ωt) + (v0/ω)·sinh(ωt)，返回 (x, ẋ)。"""
    c, s = np.cosh(w * t), np.sinh(w * t)
    x = p + (x0 - p) * c + v0 / w * s
    v = (x0 - p) * w * s + v0 * c
    return float(x), float(v)

**函数 `capture_point`**：捕获点 ξ = x + ẋ/ω：把 ZMP（脚）放在这里，质心正好停在它正上方。

In [ ]:
def capture_point(x, v, w):
    """捕获点 ξ = x + ẋ/ω：把 ZMP（脚）放在这里，质心正好停在它正上方。"""
    return x + v / w

**函数 `zmp_from_accel`**：由质心加速度反推 ZMP：p = x − ẍ/ω²。

In [ ]:
def zmp_from_accel(x, xdd, w):
    """由质心加速度反推 ZMP：p = x − ẍ/ω²。"""
    return x - xdd / w**2

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第二步 LIPM")
w = omega(0.8)
print(f"质心高 0.8 m → ω = √(9.8/0.8) = {w:.3f} 1/s")
print("想要 ẍ = 1 m/s²、质心在 0 → ZMP 要在", round(zmp_from_accel(0.0, 1.0, w), 4), "m")

In [ ]:
print("\n第三步 发散")
for t in (0.1, 0.3, 0.5):
    x, v = lipm_state(0.05, 0.0, 0.0, w, t)
    print(f"质心比 ZMP 偏 5 cm、静止起步，t={t}s：x = {x:.4f} m，ẋ = {v:.4f} m/s")

In [ ]:
print("\n第四步 捕获点")
xi = capture_point(0.0, 0.5, w)
print(f"质心在 0、速度 0.5 m/s → 捕获点 ξ = {xi:.4f} m")
x, v = lipm_state(0.0, 0.5, xi, w, 2.0)
print(f"脚踩在 ξ 上，2 s 后：x = {x:.4f}，ẋ = {v:.4f}（停在脚正上方）")

## 3. 代码里才看得到的

- 踩在捕获点上 2 秒后速度是 0.0005 而不是精确的 0：解里还剩一个按 $e^{-\omega t}$ 衰减的项（$e^{-7} \approx 0.0009$），时间越长越接近 0。

## 4. 自己改一改

- 把脚踩在捕获点前面 2 cm，看质心会往回倒还是继续向前。
- 打印 t = 0 到 1 s 的 x(t)，看 cosh 的形状。

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 公交车上压前脚，地面托你那一点叫 ZMP，只能在脚底；
> 人压成一个点，质心在 ZMP 哪边就往哪边加速；
> 歪了按 cosh 倒，脚底挪不够就得迈步；
> 位置加速度除以 ω，踩在捕获点就停住。